In [1]:
import duckdb
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, DoubleType, BooleanType, TimestampType
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
spark = SparkSession.builder.appName("Companies").getOrCreate()

In [5]:
# Connect to the DuckDB database
duck_conn = duckdb.connect(database='/home/nvkhoa14/stock-data-engineering/datawarehouse.duckdb')
query = 'SELECT * FROM dim_companies'

# Fetch data from DuckDB and convert to Pandas DataFrame
pandas_df = duck_conn.execute(query).fetchdf()

duck_conn.close()

In [6]:
# Convert Pandas DataFrame to PySpark DataFrame
spark_df_companies = spark.createDataFrame(pandas_df)

print(spark_df_companies.count())

# Show the Spark DataFrame
spark_df_companies.show()

26/03/16 09:51:12 WARN TaskSetManager: Stage 0 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


20807


26/03/16 09:51:16 WARN TaskSetManager: Stage 3 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


+----------+--------------------+--------------------+--------------+-------------------+--------------------+----------------+--------------------+---------------------+-------------------+---------------------+-----------------------+--------------------+--------------------+
|company_id|        company_name|  company_time_stamp|company_ticket|company_is_delisted|    company_category|company_currency|    company_location|company_exchange_name|company_region_name|company_industry_name|company_industry_sector|company_sic_industry|  company_sic_sector|
+----------+--------------------+--------------------+--------------+-------------------+--------------------+----------------+--------------------+---------------------+-------------------+---------------------+-----------------------+--------------------+--------------------+
|         1|ADMIRALTY BANCORP...|2026-03-11 09:18:...|          AAAB|               true|Domestic Common S...|             USD|      Florida; U.S.A|               

In [7]:
# Tạo Window specification
windowSpec = Window.partitionBy("company_ticket").orderBy(spark_df_companies["company_time_stamp"].desc())

# Sử dụng row_number để đánh số thứ tự và giữ lại hàng cuối cùng
df_with_row_num = spark_df_companies.withColumn("row_num", F.row_number().over(windowSpec))

# Lọc để giữ lại hàng cuối cùng trong mỗi nhóm
spark_df_companies_filtered = df_with_row_num.filter(df_with_row_num.row_num == 1).drop("row_num")

# Hiển thị DataFrame kết quả
print(spark_df_companies_filtered.count())
spark_df_companies_filtered.show()

26/03/16 09:51:41 WARN TaskSetManager: Stage 4 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


20807


26/03/16 09:51:50 WARN TaskSetManager: Stage 10 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


+----------+--------------------+--------------------+--------------+-------------------+--------------------+----------------+--------------------+---------------------+-------------------+---------------------+-----------------------+--------------------+--------------------+
|company_id|        company_name|  company_time_stamp|company_ticket|company_is_delisted|    company_category|company_currency|    company_location|company_exchange_name|company_region_name|company_industry_name|company_industry_sector|company_sic_industry|  company_sic_sector|
+----------+--------------------+--------------------+--------------+-------------------+--------------------+----------------+--------------------+---------------------+-------------------+---------------------+-----------------------+--------------------+--------------------+
|     14055|          ALCOA CORP|2026-03-11 09:18:...|            AA|              false|Domestic Common S...|             USD| Pennsylvania; U.S.A|               

In [8]:
# Thống kê số lượng công ty ở từng sàn giao dịch
print("Thống kê số lượng công ty ở từng sàn giao dịch:")
spark_df_companies_filtered\
    .groupBy("company_exchange_name")\
    .agg(F.count("*").alias("count"))\
    .withColumnRenamed("company_exchange_name", "Sàn giao dịch")\
    .withColumnRenamed("count", "Số lượng mã niêm yết")\
    .show()

Thống kê số lượng công ty ở từng sàn giao dịch:


26/03/16 10:03:36 WARN TaskSetManager: Stage 13 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


+-------------+--------------------+
|Sàn giao dịch|Số lượng mã niêm yết|
+-------------+--------------------+
|         NYSE|                6757|
|       NASDAQ|               14050|
+-------------+--------------------+



In [9]:
# Thống kê số lượng công ty đã và chưa bị delisted
print("Thống kê số lượng công ty bị hủy niêm yết:")
spark_df_companies_filtered\
    .groupBy("company_is_delisted")\
    .agg(F.count("*").alias("count"))\
    .withColumnRenamed("company_is_delisted", "Đã bị hủy niêm yết")\
    .withColumnRenamed("count", "Số lượng")\
    .show()

Thống kê số lượng công ty bị hủy niêm yết:


26/03/16 10:03:53 WARN TaskSetManager: Stage 19 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


+------------------+--------+
|Đã bị hủy niêm yết|Số lượng|
+------------------+--------+
|              true|   14498|
|             false|    6309|
+------------------+--------+



In [10]:
# Thống kê số lượng công ty theo quốc gia
print("Thống kê số lượng công ty theo quốc gia:")
spark_df_companies_filtered\
    .withColumn("Quóc gia", F.element_at(F.split(spark_df_companies_filtered["company_location"], "; "), -1))\
    .drop("company_location")\
    .groupBy("Quóc gia")\
    .agg(F.count("*").alias("count"))\
    .orderBy(F.col("count").desc())\
    .withColumnRenamed("count", "Số lượng công ty")\
    .show()

Thống kê số lượng công ty theo quốc gia:


26/03/16 10:04:10 WARN TaskSetManager: Stage 25 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


+--------------+----------------+
|      Quóc gia|Số lượng công ty|
+--------------+----------------+
|         U.S.A|           17322|
|         China|             597|
|        Canada|             550|
|United Kingdom|             276|
|        Israel|             238|
|Cayman Islands|             229|
|       Bermuda|             184|
|     Hong Kong|             155|
|     Singapore|             121|
|       Ireland|              83|
|        Brazil|              78|
|     Australia|              67|
|   Netherlands|              64|
|       Germany|              57|
|         Japan|              53|
|    Luxembourg|              53|
|        Taiwan|              49|
|        Greece|              48|
|   Switzerland|              48|
|        Mexico|              47|
+--------------+----------------+
only showing top 20 rows



In [11]:
# Thống kê số lượng công ty theo ngành công nghiệp
print("Thống kê số lượng công ty theo ngành công nghiệp:")
spark_df_companies_filtered\
    .groupBy("company_industry_sector")\
    .agg(F.count("*").alias("count"))\
    .orderBy(F.col("count").desc())\
    .withColumnRenamed("company_industry_sector", "Industry")\
    .withColumnRenamed("count", "Số lượng công ty")\
    .show()

Thống kê số lượng công ty theo ngành công nghiệp:


26/03/16 10:04:26 WARN TaskSetManager: Stage 31 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+----------------+
|            Industry|Số lượng công ty|
+--------------------+----------------+
|         Industrials|            4163|
|          Technology|            3557|
|          Healthcare|            3095|
|  Financial Services|            2974|
|   Consumer Cyclical|            1919|
|Communication Ser...|            1178|
|         Real Estate|            1027|
|     Basic Materials|             863|
|              Energy|             858|
|  Consumer Defensive|             779|
|           Utilities|             394|
+--------------------+----------------+



In [12]:
# Thống kê số lượng công ty theo ngành công nghiệp SIC
print("Thống kê số lượng công ty theo ngành công nghiệp SIC:")
spark_df_companies_filtered\
    .groupBy("company_sic_sector")\
    .agg(F.count("*").alias("count"))\
    .orderBy(F.col("count").desc())\
    .withColumnRenamed("company_sic_sector", "SIC Industry")\
    .withColumnRenamed("count", "Số lượng công ty")\
    .show()

Thống kê số lượng công ty theo ngành công nghiệp SIC:


26/03/16 10:04:37 WARN TaskSetManager: Stage 37 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+----------------+
|        SIC Industry|Số lượng công ty|
+--------------------+----------------+
|       Manufacturing|            6653|
|Finance Insurance...|            6194|
|            Services|            3976|
|Transportation Co...|            1667|
|        Retail Trade|             891|
|              Mining|             735|
|     Wholesale Trade|             435|
|        Construction|             183|
|Agriculture Fores...|              73|
+--------------------+----------------+



In [13]:
# Thống kê số lượng công ty theo loại hình công ty
print("Thống kê số lượng công ty theo loại hình công ty:")
spark_df_companies_filtered\
    .groupBy("company_category")\
    .agg(F.count("*").alias("count"))\
    .orderBy(F.col("count").desc())\
    .withColumnRenamed("company_category", "Category")\
    .withColumnRenamed("count", "Số lượng công ty")\
    .show()

Thống kê số lượng công ty theo loại hình công ty:


26/03/16 10:04:48 WARN TaskSetManager: Stage 43 contains a task of very large size (2183 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+----------------+
|            Category|Số lượng công ty|
+--------------------+----------------+
|Domestic Common S...|           12445|
|Domestic Common S...|            2003|
|    ADR Common Stock|            1650|
|Domestic Common S...|            1443|
|Domestic Common S...|            1190|
|Domestic Preferre...|            1085|
|Canadian Common S...|             261|
|ADR Common Stock ...|             251|
|ADR Common Stock ...|             218|
|ADR Common Stock ...|             147|
| ADR Preferred Stock|              91|
|Canadian Common S...|              13|
|Canadian Common S...|               5|
|Canadian Preferre...|               3|
|Canadian Common S...|               2|
+--------------------+----------------+



### 2. Quan sát và phân tích Fact Candles

In [16]:
# Connect to the DuckDB database
duck_conn = duckdb.connect(database='/home/nvkhoa14/stock-data-engineering/datawarehouse.duckdb')
query = 'SELECT * FROM fact_candles'

# Fetch data from DuckDB and convert to Pandas DataFrame
pandas_df = duck_conn.execute(query).fetchdf()

duck_conn.close()